# LeetCode 3539 — Find Sum of Array Product of Magical Sequences bằng Python

- Link: https://leetcode.com/problems/find-sum-of-array-product-of-magical-sequences/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp** (hệ số đa thức – multinomial), **DP theo bit có nhớ (carry)**, **nghịch đảo modulo**, chuyển "dãy có thứ tự" thành "bộ đếm số lần chọn"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def magicalSum(self, m: int, k: int, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Đổi bài toán đếm **dãy có thứ tự** `seq` thành chọn **số lần** `c_i` mỗi chỉ số xuất hiện, nhân với **hệ số đa thức** `m! / Π c_i!`.
2. Hiểu `2^{seq[0]} + … + 2^{seq[m-1]} = Σ c_i·2^i` và số bit 1 của tổng này được tính bằng **cộng có nhớ** từ bit thấp lên.
3. Thiết kế DP `dp[i][used][carry][bits]` xử lý từng chỉ số (= từng bit) với **phần nhớ** mang sang bit sau.
4. Dùng **nghịch đảo modulo** (`pow(x, MOD − 2, MOD)`) để chia cho `c!` trong trường modulo nguyên tố.
5. Kiểm chứng bằng vét cạn mọi dãy với `n, m` nhỏ.

## 2. Đề bài & ví dụ

Cho `m`, `k` và mảng `nums` (độ dài `n`). Dãy số nguyên `seq` gọi là **kỳ diệu** nếu:

- `seq` có độ dài `m`, mỗi phần tử `0 ≤ seq[i] < n` (được lặp lại);
- biểu diễn nhị phân của `2^{seq[0]} + 2^{seq[1]} + … + 2^{seq[m−1]}` có **đúng `k` bit 1**.

**Tích** của dãy: `nums[seq[0]] · nums[seq[1]] · … · nums[seq[m−1]]`. Trả về **tổng các tích** trên mọi dãy kỳ diệu, modulo `10^9 + 7`.

**Ví dụ 1:** `m = 5, k = 5, nums = [1,10,100,10000,1000000]` → `991600007` (mọi hoán vị của `[0,1,2,3,4]`, mỗi tích `10^13`).

**Ví dụ 2:** `m = 2, k = 2, nums = [5,4,3,2,1]` → `170` (mọi cặp chỉ số khác nhau có thứ tự).

**Ví dụ 3:** `m = 1, k = 1, nums = [28]` → `28`.

**Ràng buộc:** `1 ≤ k ≤ m ≤ 30`, `1 ≤ n ≤ 50`, `1 ≤ nums[i] ≤ 10^8`.

In [1]:
from typing import List
MOD = 10**9 + 7
examples = [
    (5, 5, [1, 10, 100, 10000, 1000000], 991600007),
    (2, 2, [5, 4, 3, 2, 1], 170),
    (1, 1, [28], 28),
]
for m, k, nums, exp in examples:
    print(m, k, nums, '-> kỳ vọng', exp)

5 5 [1, 10, 100, 10000, 1000000] -> kỳ vọng 991600007
2 2 [5, 4, 3, 2, 1] -> kỳ vọng 170
1 1 [28] -> kỳ vọng 28


## 3. Kiến thức nền

### 3.1. Đếm bit 1

`bin(x).count('1')` hoặc `x.bit_count()` (Python ≥ 3.10).

In [2]:
seq = [0, 0, 2]
s = sum(1 << x for x in seq)
print(seq, '->', s, bin(s), 'số bit 1 =', bin(s).count('1'))

[0, 0, 2] -> 6 0b110 số bit 1 = 2


### 3.2. Hệ số đa thức (multinomial)

Số dãy có thứ tự độ dài `m` trong đó chỉ số `i` xuất hiện đúng `c_i` lần (`Σ c_i = m`) là `m! / (c_0! c_1! … c_{n−1}!)`. Mọi dãy như vậy có **cùng** tích `Π nums[i]^{c_i}` và **cùng** tổng `Σ c_i 2^i`.

In [3]:
from math import factorial
from itertools import permutations
c = {0: 2, 2: 1}
print('công thức:', factorial(3) // (factorial(2) * factorial(1)),
      '| liệt kê:', len(set(permutations([0, 0, 2]))))

công thức: 3 | liệt kê: 3


### 3.3. Nghịch đảo modulo

`MOD` nguyên tố ⇒ theo định lý Fermat nhỏ `a^{−1} ≡ a^{MOD−2}`. Dựng sẵn `fact[i]` và `inv_fact[i]`.

In [4]:
fact = [1] * 31
for i in range(1, 31):
    fact[i] = fact[i - 1] * i % MOD
inv_fact = [pow(f, MOD - 2, MOD) for f in fact]
print(fact[5], inv_fact[5], fact[5] * inv_fact[5] % MOD)

120 808333339 1


### 3.4. Cộng nhiều lũy thừa của 2 = cộng có nhớ theo từng bit

`Σ c_i 2^i`: xử lý từ bit 0. Tại bit `i`, tổng "cột" là `t = c_i + carry`; bit kết quả là `t & 1`, phần nhớ sang bit `i + 1` là `t >> 1`. Sau bit cuối, phần nhớ còn lại đóng góp `popcount(carry)` bit nữa.

In [5]:
def popcount_by_columns(counts):
    carry, bits = 0, 0
    for i, ci in enumerate(counts):
        t = ci + carry
        bits += t & 1
        carry = t >> 1
        print(f"bit {i}: c={ci}, nhớ vào={t - ci}, tổng cột={t} -> bit {t & 1}, nhớ ra {carry}")
    return bits + bin(carry).count('1')

counts = [3, 1, 2]           # 3·1 + 1·2 + 2·4 = 13 = 0b1101
print('popcount =', popcount_by_columns(counts), '| kiểm tra:', bin(3 + 2 + 8).count('1'))

bit 0: c=3, nhớ vào=0, tổng cột=3 -> bit 1, nhớ ra 1
bit 1: c=1, nhớ vào=1, tổng cột=2 -> bit 0, nhớ ra 1
bit 2: c=2, nhớ vào=1, tổng cột=3 -> bit 1, nhớ ra 1
popcount = 3 | kiểm tra: 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `m = 2, k = 2`. Tổng `2^a + 2^b` có 2 bit 1 ⇔ `a ≠ b` (nếu `a = b` thì `2^{a+1}` chỉ có 1 bit). Vậy đáp án = `Σ_{a ≠ b} nums[a]·nums[b] = (Σ nums)^2 − Σ nums^2 = 15² − 55 = 170` ✓.

Tổng quát: thay vì liệt kê `n^m` dãy, chọn **bộ đếm** `(c_0, …, c_{n−1})`:

`đáp án = Σ_{c: Σc = m, popcount(Σ c_i 2^i) = k} m! · Π (nums[i]^{c_i} / c_i!)`.

Biểu thức trong tổng **tách được** theo từng `i` (tích), chỉ có điều kiện popcount là "dính" giữa các `i` — nhưng điều kiện đó xử lý được tuần tự từ bit thấp lên với **phần nhớ** ⇒ DP.

In [6]:
nums = [5, 4, 3, 2, 1]
print('công thức nhanh cho m = k = 2:', sum(nums) ** 2 - sum(x * x for x in nums))

công thức nhanh cho m = k = 2: 170


## 5. Cách 1 — Brute Force: liệt kê mọi dãy

`n^m` dãy; với mỗi dãy kiểm popcount và cộng tích.

In [7]:
from itertools import product

def magical_brute(m, k, nums):
    n = len(nums)
    total = 0
    for seq in product(range(n), repeat=m):
        if bin(sum(1 << x for x in seq)).count('1') == k:
            p = 1
            for x in seq:
                p *= nums[x]
            total += p
    return total % MOD

for m, k, nums, exp in examples:
    print(magical_brute(m, k, nums), '| kỳ vọng', exp)

991600007 | kỳ vọng 991600007
170 | kỳ vọng 170
28 | kỳ vọng 28


**Độ phức tạp:** `O(n^m · m)` — với `n = 50, m = 30` là `50^{30}`: không tưởng.

**Nút thắt:** thứ tự trong dãy không ảnh hưởng tổng lẫn tích ⇒ đếm theo **bộ đếm** và nhân hệ số đa thức.

## 6. Ý tưởng thuật toán / bước đột phá

DP xử lý chỉ số `i = 0, 1, …, n−1` (cũng là bit `i`):

`dp[(used, carry, bits)]` = tổng trọng số `Π_{j<i} nums[j]^{c_j} / c_j!` trên mọi cách chọn `c_0..c_{i−1}` sao cho đã dùng `used = Σ c_j` phần tử, phần nhớ sang bit `i` là `carry`, và đã "chốt" `bits` bit 1 ở các bit `< i`.

Chuyển trạng thái với `c = c_i ∈ [0, m − used]`:

- `t = c + carry`, `bits' = bits + (t & 1)`, `carry' = t >> 1`, `used' = used + c`;
- trọng số nhân thêm `nums[i]^c / c!`.

Kết thúc: `đáp án = m! · Σ dp[(m, carry, bits)]` với `bits + popcount(carry) = k`.

Cắt nhánh: `bits > k` thì bỏ (bit 1 đã chốt không mất đi). `carry ≤ used/2 ≤ 15`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Liệt kê bộ đếm `c` + hệ số đa thức

Duyệt mọi bộ `(c_0..c_{n−1})` với tổng `m` (số bộ = `C(m + n − 1, n − 1)`), cộng `m!/Πc_i! · Π nums^{c_i}` nếu popcount đúng. Nhanh hơn `n^m` nhiều nhưng vẫn mũ.

In [8]:
def compositions(m, n):
    if n == 1:
        yield (m,)
        return
    for c in range(m + 1):
        for rest in compositions(m - c, n - 1):
            yield (c,) + rest

def magical_counts(m, k, nums):
    n = len(nums)
    total = 0
    for cs in compositions(m, n):
        if bin(sum(c << i for i, c in enumerate(cs))).count('1') != k:
            continue
        coef = factorial(m)
        for c in cs:
            coef //= factorial(c)
        p = coef
        for i, c in enumerate(cs):
            p *= nums[i] ** c
        total += p
    return total % MOD

for m, k, nums, exp in examples:
    print(magical_counts(m, k, nums), '| kỳ vọng', exp)

991600007 | kỳ vọng 991600007
170 | kỳ vọng 170
28 | kỳ vọng 28


**Độ phức tạp:** `C(m + n − 1, n − 1)` bộ — với `n = 50, m = 30` khoảng `10^{20}`: vẫn quá lớn.

### 7.2. Cách 3 — DP có nhớ (top-down) theo `(i, used, carry, bits)`

In [9]:
from functools import lru_cache

def magical_memo(m, k, nums):
    n = len(nums)
    coef = [[pow(x, c, MOD) * inv_fact[c] % MOD for c in range(m + 1)] for x in nums]

    @lru_cache(maxsize=None)
    def go(i, used, carry, bits):
        if bits > k:
            return 0
        if i == n:
            return 1 if used == m and bits + bin(carry).count('1') == k else 0
        res = 0
        for c in range(m - used + 1):
            t = c + carry
            res += coef[i][c] * go(i + 1, used + c, t >> 1, bits + (t & 1))
        return res % MOD

    return go(0, 0, 0, 0) * fact[m] % MOD

for m, k, nums, exp in examples:
    print(magical_memo(m, k, nums), '| kỳ vọng', exp)

991600007 | kỳ vọng 991600007
170 | kỳ vọng 170
28 | kỳ vọng 28


### 7.3. Cách 4 — DP bottom-up với từ điển trạng thái (lời giải chính)

Chỉ giữ các trạng thái **đạt được** (dict), lớp theo `i`.

In [10]:
def magical_dp(m, k, nums):
    dp = {(0, 0, 0): 1}
    for x in nums:
        pw = [1] * (m + 1)
        for c in range(1, m + 1):
            pw[c] = pw[c - 1] * x % MOD
        cf = [pw[c] * inv_fact[c] % MOD for c in range(m + 1)]
        nd = {}
        for (used, carry, bits), v in dp.items():
            for c in range(m - used + 1):
                t = c + carry
                nb_ = bits + (t & 1)
                if nb_ > k:
                    continue
                key = (used + c, t >> 1, nb_)
                nd[key] = (nd.get(key, 0) + v * cf[c]) % MOD
        dp = nd
    ans = 0
    for (used, carry, bits), v in dp.items():
        if used == m and bits + bin(carry).count('1') == k:
            ans += v
    return ans * fact[m] % MOD

for m, k, nums, exp in examples:
    print(magical_dp(m, k, nums), '| kỳ vọng', exp)

991600007 | kỳ vọng 991600007
170 | kỳ vọng 170
28 | kỳ vọng 28


**Độ phức tạp:** số trạng thái mỗi lớp ≤ `(m+1) · (m/2+1) · (k+1)`; mỗi trạng thái thử ≤ `m + 1` giá trị `c` ⇒ `O(n · m^3 · k / 2)` trường hợp xấu (~vài triệu với giới hạn đề, nhờ cắt nhánh còn ít hơn).

### 7.4. Cách 5 — Lời giải đóng cho `m = k`

Khi `m = k`, `popcount = m` ⇔ mọi chỉ số **khác nhau** (lặp chỉ số sẽ tạo nhớ và làm giảm số bit). Đáp án = `m! · e_m(nums)` với `e_m` là **đa thức đối xứng sơ cấp** bậc `m` — tính bằng DP `O(n·m)`. Đây là trường hợp đặc biệt hữu ích để kiểm tra (ví dụ 1, 2, 3 đều có `m = k`).

In [11]:
def magical_m_equals_k(m, nums):
    e = [1] + [0] * m                 # e[j] = đa thức đối xứng sơ cấp bậc j
    for x in nums:
        for j in range(m, 0, -1):
            e[j] = (e[j] + e[j - 1] * x) % MOD
    return e[m] * fact[m] % MOD

for m, k, nums, exp in examples:
    print(magical_m_equals_k(m, nums), '| kỳ vọng', exp)

991600007 | kỳ vọng 991600007
170 | kỳ vọng 170
28 | kỳ vọng 28


## 8. TRACE từng bước

Trace cách 4 với `m = 3, k = 2, nums = [2, 3]` (nhỏ để in hết trạng thái). In các trạng thái `(used, carry, bits) → trọng số` sau mỗi chỉ số và đóng góp cuối cùng.

In [12]:
from fractions import Fraction

def trace(m, k, nums):
    dp = {(0, 0, 0): Fraction(1)}
    for i, x in enumerate(nums):
        nd = {}
        for (used, carry, bits), v in dp.items():
            for c in range(m - used + 1):
                t = c + carry
                key = (used + c, t >> 1, bits + (t & 1))
                if key[2] > k:
                    continue
                nd[key] = nd.get(key, 0) + v * Fraction(x ** c, factorial(c))
        dp = nd
        print(f"sau chỉ số {i} (nums={x}):")
        for key in sorted(dp):
            print(f"    (used, carry, bits) = {key}: trọng số {dp[key]}")
    ans = 0
    for (used, carry, bits), v in dp.items():
        if used == m and bits + bin(carry).count('1') == k:
            contrib = v * factorial(m)
            print(f"  chọn {(used, carry, bits)}: {v} · {m}! = {contrib}")
            ans += contrib
    print('Đáp án (chưa modulo) =', ans, '| vét cạn =', magical_brute(m, k, nums))
    return ans

trace(3, 2, [2, 3])

sau chỉ số 0 (nums=2):
    (used, carry, bits) = (0, 0, 0): trọng số 1
    (used, carry, bits) = (1, 0, 1): trọng số 2
    (used, carry, bits) = (2, 1, 0): trọng số 2
    (used, carry, bits) = (3, 1, 1): trọng số 4/3
sau chỉ số 1 (nums=3):
    (used, carry, bits) = (0, 0, 0): trọng số 1
    (used, carry, bits) = (1, 0, 1): trọng số 5
    (used, carry, bits) = (2, 0, 1): trọng số 2
    (used, carry, bits) = (2, 0, 2): trọng số 6
    (used, carry, bits) = (2, 1, 0): trọng số 9/2
    (used, carry, bits) = (3, 0, 2): trọng số 4/3
    (used, carry, bits) = (3, 1, 0): trọng số 6
    (used, carry, bits) = (3, 1, 1): trọng số 27/2
  chọn (3, 1, 1): 27/2 · 3! = 81
  chọn (3, 0, 2): 4/3 · 3! = 8
Đáp án (chưa modulo) = 89 | vét cạn = 89


Fraction(89, 1)

Kiểm tra tay: các dãy độ dài 3 trên `{0, 1}` có popcount 2: bộ đếm `(c0, c1)` ∈ `{(3,0): 3 = 0b11 ✓, (2,1): 2+2 = 4 ✗, (1,2): 1+4 = 5 ✓, (0,3): 6 = 0b110 ✓}`. Đóng góp: `1·2^3 = 8`, `3·2·3^2 = 54`, `1·3^3 = 27` ⇒ `89`.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def magicalSum(self, m: int, k: int, nums: List[int]) -> int:
        MOD = 10**9 + 7
        fact = [1] * (m + 1)
        for i in range(1, m + 1):
            fact[i] = fact[i - 1] * i % MOD
        inv_fact = [pow(f, MOD - 2, MOD) for f in fact]

        dp = {(0, 0, 0): 1}
        for x in nums:
            cf = [1] * (m + 1)
            p = 1
            for c in range(1, m + 1):
                p = p * x % MOD
                cf[c] = p * inv_fact[c] % MOD
            nd = {}
            for (used, carry, bits), v in dp.items():
                for c in range(m - used + 1):
                    t = c + carry
                    nb_ = bits + (t & 1)
                    if nb_ > k:
                        continue
                    key = (used + c, t >> 1, nb_)
                    nd[key] = (nd.get(key, 0) + v * cf[c]) % MOD
            dp = nd

        ans = 0
        for (used, carry, bits), v in dp.items():
            if used == m and bits + bin(carry).count('1') == k:
                ans += v
        return ans % MOD * fact[m] % MOD

for m, k, nums, exp in examples:
    got = Solution().magicalSum(m, k, nums)
    print(got, 'OK' if got == exp else 'SAI')

991600007 OK
170 OK
28 OK


## 10. Vì sao thuật toán đúng

1. **Gom theo bộ đếm:** ánh xạ dãy → bộ đếm `c` là toàn ánh; mỗi bộ đếm có đúng `m!/Πc_i!` dãy, cùng tích `Π nums[i]^{c_i}` và cùng tổng `Σ c_i 2^i` ⇒ tổng theo dãy = tổng theo bộ đếm có trọng số multinomial.
2. **Tách tích:** `m!/Πc_i! · Π nums[i]^{c_i} = m! · Π (nums[i]^{c_i}/c_i!)` ⇒ trọng số nhân dần theo từng `i`.
3. **Cộng có nhớ đúng:** bit `i` của `Σ c_j 2^j` chỉ phụ thuộc `c_i` và phần nhớ từ các bit thấp hơn; phần nhớ được truyền đúng bằng `t >> 1`. Sau bit `n − 1`, phần nhớ còn lại `carry` biểu diễn phần cao `carry · 2^n`, góp `popcount(carry)` bit.
4. **Cắt nhánh an toàn:** bit đã chốt không thay đổi về sau ⇒ `bits > k` không bao giờ quay về `k`.
5. **Modulo:** mọi phép nhân/cộng trong trường `Z_MOD`; chia cho `c!` bằng nhân nghịch đảo (MOD nguyên tố, `c ≤ 30 < MOD`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 50`, `m ≤ 30`, `k ≤ m`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy | `O(n^m · m)` | `O(m)` |
| 2. Liệt kê bộ đếm | `O(C(m+n−1, n−1) · n)` | `O(n)` |
| 3. DP có nhớ | `O(n · m · (m/2) · k · m)` | số trạng thái |
| 4. DP dict | như 3, chỉ trạng thái đạt được | `O(m^2 k)` mỗi lớp |
| 5. `m = k`: đa thức đối xứng | `O(n m)` | `O(m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [14]:
impls = {
    'memo': magical_memo,
    'dp': magical_dp,
    'submit': lambda m, k, nums: Solution().magicalSum(m, k, nums),
}
edge = [
    (2, 1, [7], 49),                  # [0,0]: 2+... = 2^1 -> 1 bit
    (3, 1, [7], 0),                   # 3·2^0 = 3 = 0b11 -> 2 bit
    (4, 1, [2], 16),                  # 4 = 0b100
    (3, 3, [1, 1], 0),                # cần 3 chỉ số khác nhau nhưng chỉ có 2
    (2, 1, [3, 5], 3 * 3 + 5 * 5),    # chỉ dãy lặp cùng chỉ số
]
for m, k, nums, exp in examples + edge:
    for name, f in impls.items():
        got = f(m, k, nums)
        assert got == exp % MOD, (name, m, k, nums, got, exp)
    if len(nums) ** m <= 10**6:
        assert magical_brute(m, k, nums) == exp % MOD
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn và với cách 2

In [15]:
import random
random.seed(3539)
cnt = 0
for _ in range(300):
    n = random.randint(1, 4)
    m = random.randint(1, 6)
    k = random.randint(1, m)
    nums = [random.randint(1, 10**8) for _ in range(n)]
    exp = magical_brute(m, k, nums)
    for name, f in impls.items():
        assert f(m, k, nums) == exp, (name, m, k, nums)
    assert magical_counts(m, k, nums) == exp
    if m == k:
        assert magical_m_equals_k(m, nums) == exp
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 300 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 50`, `m = 30`, nhiều `k`)

In [16]:
import time
random.seed(30)
nums = [random.randint(1, 10**8) for _ in range(50)]
for k in [1, 5, 15, 25, 30]:
    t = time.time(); a = Solution().magicalSum(30, k, nums); t1 = time.time() - t
    if k == 30:
        assert a == magical_m_equals_k(30, nums)
    print(f"m=30, k={k:>2}: {a} | {t1:.2f}s")

m=30, k= 1: 947381047 | 0.05s


m=30, k= 5: 408530717 | 0.17s


m=30, k=15: 850062586 | 0.24s


m=30, k=25: 64278344 | 0.26s


m=30, k=30: 933916800 | 0.25s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên hệ số đa thức (coi dãy như tập/multiset)

In [17]:
def wrong_no_multinomial(m, k, nums):
    n = len(nums)
    total = 0
    for cs in compositions(m, n):
        if bin(sum(c << i for i, c in enumerate(cs))).count('1') == k:
            p = 1
            for i, c in enumerate(cs):
                p *= nums[i] ** c
            total += p                 # SAI: thiếu m! / Π c!
    return total % MOD

m, k, nums, exp = examples[1]
print('sai :', wrong_no_multinomial(m, k, nums), '| đúng:', Solution().magicalSum(m, k, nums))

sai : 85 | đúng: 170


**Vì sao sai:** `seq` là dãy **có thứ tự**: `[0, 1]` và `[1, 0]` là hai dãy khác nhau.

### Lỗi 2 — Đếm bit 1 bằng "số chỉ số khác nhau" (bỏ qua phần nhớ)

In [18]:
def wrong_distinct_count(m, k, nums):
    n = len(nums)
    total = 0
    for seq in product(range(n), repeat=m):
        if len(set(seq)) == k:          # SAI: [0,0,1] -> 1+1+2 = 4 chỉ có 1 bit
            p = 1
            for x in seq:
                p *= nums[x]
            total += p
    return total % MOD

print('sai :', wrong_distinct_count(3, 2, [2, 3]), '| đúng:', Solution().magicalSum(3, 2, [2, 3]))

sai : 90 | đúng: 89


**Vì sao sai:** chỉ số lặp tạo **phép nhớ** (`2^a + 2^a = 2^{a+1}`), có thể làm số bit tăng hoặc giảm theo cách phức tạp.

### Lỗi 3 — Quên phần nhớ còn lại sau bit cuối

In [19]:
def wrong_forget_carry(m, k, nums):
    dp = {(0, 0, 0): 1}
    for x in nums:
        nd = {}
        for (used, carry, bits), v in dp.items():
            for c in range(m - used + 1):
                t = c + carry
                key = (used + c, t >> 1, bits + (t & 1))
                nd[key] = (nd.get(key, 0) + v * pow(x, c, MOD) * inv_fact[c]) % MOD
        dp = nd
    ans = sum(v for (used, carry, bits), v in dp.items() if used == m and bits == k)   # SAI
    return ans * fact[m] % MOD

print('sai :', wrong_forget_carry(2, 1, [7]), '| đúng:', Solution().magicalSum(2, 1, [7]))

sai : 0 | đúng: 49


**Vì sao sai:** `[0, 0]` cho `2 = 0b10`: bit 0 bằng 0, phần nhớ 1 sang bit 1 (vượt quá chỉ số cuối) — phải cộng `popcount(carry)`.

### Lỗi 4 — Chia `c!` bằng phép chia nguyên sau khi đã lấy modulo

In [20]:
x = factorial(20) % MOD
print('chia nguyên sau mod:', x // factorial(10) % MOD, '| đúng (nghịch đảo):', x * inv_fact[10] % MOD,
      '| thật:', factorial(20) // factorial(10) % MOD)

chia nguyên sau mod: 40 | đúng (nghịch đảo): 442568110 | thật: 442568110


**Vì sao sai:** sau khi lấy modulo, số không còn chia hết; phải nhân **nghịch đảo modulo**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Dãy có thứ tự, chỉ quan tâm số lần mỗi phần tử" ⇒ đếm theo bộ đếm × **hệ số đa thức**; tách `m!/Πc!` thành `m! · Π(1/c!)` để DP nhân dần.
- "Popcount của tổng các lũy thừa 2" ⇒ **cộng có nhớ theo bit**, trạng thái = phần nhớ (nhỏ, ≤ m/2).
- Modulo nguyên tố + chia ⇒ `pow(x, MOD − 2, MOD)`.
- Trường hợp đặc biệt (`m = k` ⇒ chỉ số phân biệt) là công cụ kiểm thử tốt.

## 15. Biến thể của bài

1. Đếm **số dãy** kỳ diệu (không có trọng số) ⇒ đặt mọi `nums[i] = 1`.
2. Điều kiện "popcount ≤ k" ⇒ cộng mọi trạng thái cuối có `bits + popcount(carry) ≤ k`.
3. Thay lũy thừa 2 bằng lũy thừa 3 (tổng các chữ số cơ số 3) ⇒ nhớ theo cơ số 3: `t % 3`, `t // 3`.
4. Bài liên quan: đếm cách viết số dưới dạng tổng lũy thừa 2 (binary partitions), DP chữ số có nhớ.

In [21]:
# Biến thể 1 & 2
def count_magical(m, k, n):
    return Solution().magicalSum(m, k, [1] * n)

def magical_at_most(m, k, nums):
    return sum(Solution().magicalSum(m, j, nums) for j in range(1, k + 1)) % MOD

print('số dãy kỳ diệu (m=3,k=2,n=3):', count_magical(3, 2, 3),
      '| vét cạn:', sum(1 for s in product(range(3), repeat=3) if bin(sum(1 << x for x in s)).count('1') == 2))
print('popcount <= 2 (m=3, nums=[2,3]):', magical_at_most(3, 2, [2, 3]))

số dãy kỳ diệu (m=3,k=2,n=3): 15 | vét cạn: 15
popcount <= 2 (m=3, nums=[2,3]): 125


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Liệt kê dãy | `O(n^m m)` | `O(m)` | Oracle |
| 2 | Liệt kê bộ đếm | `O(C(m+n−1, n−1))` | `O(n)` | Oracle tốt hơn |
| 3 | DP có nhớ | đa thức | trạng thái | Dễ viết |
| 4 | DP dict | đa thức | `O(m^2 k)` | **Nộp bài** |
| 5 | Đa thức đối xứng (`m = k`) | `O(nm)` | `O(m)` | Trường hợp đặc biệt / kiểm thử |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `magicalSum(2, 1, [1, 1])` = ?

**Bài 2 (tính tay):** `magicalSum(3, 2, [2, 3])` (xem trace) = ?

**Bài 3 (điền TODO):** viết `multinomial(cs)` tính `(Σcs)! / Π c!` modulo `MOD`.

**Bài 4 (tự viết lại):** biến thể 3 — tổng lũy thừa **3**, điều kiện "tổng các chữ số cơ số 3 bằng `k`".

**Bài 5 (mini challenge):** chứng minh bằng code rằng `carry ≤ used / 2` luôn đúng ở mọi trạng thái DP (in ra max tỉ lệ quan sát được).

In [22]:
def multinomial_todo(cs):
    # TODO
    raise NotImplementedError

def magical_base3_todo(m, k, nums):
    # TODO
    raise NotImplementedError

def carry_bound_todo(m, k, nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dãy: `[0,0] → 2`, `[1,1] → 4` (1 bit); `[0,1], [1,0] → 3` (2 bit). (2) Chỉ hai dãy lặp. (3) `2`.

**Bài 2.** (1) Xem bảng ở mục 8. (2) `8 + 54 + 27`. (3) `89`.

**Bài 3.** (1) `fact[Σ] · Π inv_fact[c]`. (2) Modulo từng bước. (3) So với `factorial` chính xác.

**Bài 4.** (1) Chữ số cơ số 3: `t % 3`, nhớ `t // 3`. (2) Cộng `t % 3` vào tổng chữ số. (3) Cuối: cộng tổng chữ số cơ số 3 của phần nhớ.

**Bài 5.** (1) Quy nạp: `carry_{i+1} = (c_i + carry_i) // 2`. (2) `used_{i+1} = used_i + c_i`. (3) In `max(carry / used)`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
a1 = Solution().magicalSum(2, 1, [1, 1])
a2 = Solution().magicalSum(3, 2, [2, 3])
assert a1 == 2 == magical_brute(2, 1, [1, 1])
assert a2 == 89 == magical_brute(3, 2, [2, 3])

def multinomial(cs):
    r = fact[sum(cs)]
    for c in cs:
        r = r * inv_fact[c] % MOD
    return r

for cs in [(2, 1), (3, 0, 2), (5, 5, 5, 5)]:
    exact = factorial(sum(cs))
    for c in cs:
        exact //= factorial(c)
    assert multinomial(cs) == exact % MOD
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: multinomial đúng')

Bài 1: 2 | Bài 2: 89 | Bài 3: multinomial đúng


In [24]:
# Bài 4: cơ số 3
def digit_sum3(x):
    s = 0
    while x:
        s += x % 3
        x //= 3
    return s

def magical_base3(m, k, nums):
    dp = {(0, 0, 0): 1}
    for x in nums:
        cf = [pow(x, c, MOD) * inv_fact[c] % MOD for c in range(m + 1)]
        nd = {}
        for (used, carry, s), v in dp.items():
            for c in range(m - used + 1):
                t = c + carry
                ns = s + t % 3
                if ns > k:
                    continue
                key = (used + c, t // 3, ns)
                nd[key] = (nd.get(key, 0) + v * cf[c]) % MOD
        dp = nd
    ans = sum(v for (used, carry, s), v in dp.items() if used == m and s + digit_sum3(carry) == k)
    return ans * fact[m] % MOD

def base3_brute(m, k, nums):
    tot = 0
    for seq in product(range(len(nums)), repeat=m):
        if digit_sum3(sum(3 ** x for x in seq)) == k:
            p = 1
            for x in seq:
                p *= nums[x]
            tot += p
    return tot % MOD

random.seed(4)
for _ in range(150):
    n, m = random.randint(1, 3), random.randint(1, 5)
    k = random.randint(1, m)
    nums = [random.randint(1, 9) for _ in range(n)]
    assert magical_base3(m, k, nums) == base3_brute(m, k, nums)
print('Bài 4: cơ số 3 khớp vét cạn')

Bài 4: cơ số 3 khớp vét cạn


In [25]:
# Bài 5: kiểm tra carry <= used / 2
def carry_bound(m, k, nums):
    dp = {(0, 0, 0)}
    worst = 0.0
    for _ in nums:
        nd = set()
        for used, carry, bits in dp:
            for c in range(m - used + 1):
                t = c + carry
                st = (used + c, t >> 1, bits + (t & 1))
                if st[2] <= k:
                    nd.add(st)
                    if st[0]:
                        worst = max(worst, st[1] / st[0])
        dp = nd
    return worst

print('max carry/used quan sát được:', carry_bound(30, 30, [1] * 50), '(≤ 0.5)')
assert carry_bound(30, 30, [1] * 50) <= 0.5

max carry/used quan sát được: 0.5 (≤ 0.5)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dp[(used, carry, bits)] = 1 ở (0, 0, 0)
for x in nums:                 # chỉ số i = bit i
    for c in 0..m-used: t = c + carry
        -> (used + c, t >> 1, bits + (t & 1)), nhân x^c / c!
đáp án = m! · Σ dp[(m, carry, bits)] với bits + popcount(carry) == k
```

**Checklist:**

- [ ] Nhân `m!` và chia `c!` (dãy có thứ tự)?
- [ ] Popcount tính qua **cộng có nhớ**, không phải số chỉ số khác nhau?
- [ ] Cộng `popcount(carry)` sau chỉ số cuối?
- [ ] Chia bằng nghịch đảo modulo?
- [ ] Cắt nhánh `bits > k`?